In [1]:
import requests
import pandas as pd
import time

In [2]:
headers = {
    'accept': 'application/json'
}

In [ ]:
# montagem do dicionario com as informações 
def dataset_builder(registro, dataset, fonte, descricao):
    d = {
        'dataset_id':registro.get('dataset_id',''),
        'dataset_unique_id':registro.get('dataset_unique_id',''),
        'status':registro.get('status',''),
        'nome_dataset':registro.get('name',''),
        'fonte_dataset':registro.get('source',''),
        'ultimo_update_dataset':registro.get('last_updated_date',''),
        'link':dataset.get('app_url',''),
        'link_antigo':dataset.get('app_legacy_url',''),
        'data_criado':dataset.get('dates')[1].get('date','') if dataset.get('dates') else None,
        'periodicidade':dataset.get('temporal_resolution').get('periodicity','') if dataset.get('temporal_resolution') else None,
        'versoes':len(dataset.get('maintenance_information').get('version_history','')) if dataset.get('maintenance_information') else None,
        'resource_id':fonte.get('resource_id',''),
        'resource_unique_id':fonte.get('resource_unique_id',''),
        'nome_fonte':fonte.get('name',''),
        'url_fonte':fonte.get('url',''),
        'web_url_fonte':fonte.get('website_url',''),
        'formato_fonte':fonte.get('format',''),
        'publicacao_fonte': descricao.get('first_published_date',''),
        'visibilidade': descricao.get('constraints').get('security').get('classification','') if descricao.get('constraints') else None,
        'tamanho_fonte': descricao.get('distribuition').get('distribution_size','') if descricao.get('distribuition') else None
    }
    return d

In [ ]:
skip = 0

# lista para armazenar os datasets
lista_dataset = []

while True:
    # url para ver todos datasets
    url_lista = f'https://datacatalogapi.worldbank.org/ddhxext/DatasetList?$top=1000&$skip={skip}'
    resposta_lista = requests.get(url_lista, headers=headers)
    
    if resposta_lista.status_code != 200:
        continue
    
    registros = resposta_lista.json()
    
    # navegando em cada dataset
    for r in registros.get('data'):
        id = r.get('dataset_unique_id')
        
        # print('Processando dataset ', id)
        
        if not id:
            continue
        
        # obter informações complementares sobre o dataset
        url_dataset = f'https://datacatalogapi.worldbank.org/ddhxext/DatasetView?dataset_unique_id={id}'
        metadados = requests.get(url_dataset, headers=headers)
        
        if metadados.status_code != 200:
            continue
        
        data_set = metadados.json()
        
        # obter lista de recursos
        fontes = data_set.get('Resources')
        
        if not fontes:
            continue
        
        # navegando em cada recurso do dataset
        for f in fontes:
            # obtendo id do recurso
            id_fonte = f.get('resource_unique_id')
            
            # print('Lendo fonte ', id_fonte)
            
            if not id_fonte:
                continue
            
            # url para obter mais informações do recurso
            ulr_fonte = f'https://datacatalogapi.worldbank.org/ddhxext/ResourceView?resource_unique_id={id_fonte}'
            fonte_dado = requests.get(ulr_fonte, headers=headers)
            
            if fonte_dado.status_code != 200:
                continue
            
            description = fonte_dado.json()
            
            if not description:
                continue
            
            # montar o dicionario do dataset
            dict_dataset = dataset_builder(r, data_set, f, description)
            
            # armazenar o dataset na lista
            lista_dataset.append(dict_dataset)
            
            time.sleep(0.1)
        
        time.sleep(0.1)
    
    # encerrar o loop caso exceda o total de datasets
    limite = registros.get('count')
    skip += 1000
    if limite < skip:
        break

In [9]:
df = pd.DataFrame(lista_dataset)
df.to_csv('C:/Users/user/Desktop/bolsa estudo/dados_worldbank.csv', index=False, encoding='utf-8')